# 12 — Cascade Delay Analysis

Analyzes how delays propagate through aircraft rotations in the test set (Jan–Aug 2025). Validates the cascade feature design and quantifies the late aircraft domino effect.

**Input:** `dataset/merged_flights_fe_v2.parquet` (test set: Jan–Aug 2025)

**Key findings:**
- Flights with cascade_score ≥ 1 delay at significantly higher rates
- Late aircraft cascade accounts for 39.9% of all delay minutes
- Hub airports (ATL, ORD, DFW) show highest cascade propagation

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

flights = pd.read_parquet('dataset/merged_flights_fe_v2.parquet',
    columns=['FL_DATE', 'TAIL_NUM', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST',
             'DEP_HOUR', 'ARR_HOUR', 'ARR_DEL15', 'ARR_DELAY',
             'cascade_score', 'cascade_delay_minutes', 'prev_tail_arr_delay',
             'scheduled_turnaround_buffer', 'real_time_turn_gap',
             'LATE_AIRCRAFT_DELAY', 'CARRIER_DELAY', 'NAS_DELAY'])

flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
test = flights[flights['FL_DATE'] >= '2025-01-01'].copy()

print(f"Test set: {test.shape[0]:,} flights")
print(f"Unique tail numbers: {test['TAIL_NUM'].nunique():,}")
print(f"Delayed flights: {(test['ARR_DEL15'] == 1).sum():,} ({test['ARR_DEL15'].mean()*100:.1f}%)")
print("✓ Ready")

Test set: 4,519,126 flights
Unique tail numbers: 5,972
Delayed flights: 1,037,836 (23.0%)
✓ Ready


## Cascade Propagation Statistics
Quantify how prior tail delays translate into current flight delays.

In [ ]:
test = test.sort_values(['TAIL_NUM', 'FL_DATE', 'DEP_HOUR']).reset_index(drop=True)

has_prev_delay = test['prev_tail_arr_delay'].notna() & (test['prev_tail_arr_delay'] > 0)
print(f"{'=' * 60}")
print(f"CASCADE PROPAGATION ANALYSIS")
print(f"{'=' * 60}")

print(f"\nFlights where previous aircraft was late: {has_prev_delay.sum():,}")
print(f"  Of those, this flight was ALSO delayed: {(test.loc[has_prev_delay, 'ARR_DEL15'] == 1).sum():,} "
      f"({(test.loc[has_prev_delay, 'ARR_DEL15'] == 1).mean()*100:.1f}%)")
print(f"  Of those, this flight was ON TIME:      {(test.loc[has_prev_delay, 'ARR_DEL15'] == 0).sum():,} "
      f"({(test.loc[has_prev_delay, 'ARR_DEL15'] == 0).mean()*100:.1f}%)")

no_prev_delay = test['prev_tail_arr_delay'].notna() & (test['prev_tail_arr_delay'] <= 0)
print(f"\nFlights where previous aircraft was ON TIME: {no_prev_delay.sum():,}")
print(f"  Of those, this flight was delayed: {(test.loc[no_prev_delay, 'ARR_DEL15'] == 1).sum():,} "
      f"({(test.loc[no_prev_delay, 'ARR_DEL15'] == 1).mean()*100:.1f}%)")

print(f"\nPROPAGATION RATE BY INCOMING DELAY SEVERITY:")
print(f"  {'Incoming Delay':<25} {'Flights':>10} {'Propagated':>12} {'Rate':>8}")
print(f"  {'-' * 58}")

bins = [(0, 15), (15, 30), (30, 60), (60, 120), (120, 500)]
for low, high in bins:
    mask = has_prev_delay & (test['prev_tail_arr_delay'] >= low) & (test['prev_tail_arr_delay'] < high)
    count = mask.sum()
    if count > 0:
        prop_rate = test.loc[mask, 'ARR_DEL15'].mean() * 100
        print(f"  {low:>3}-{high:<3} min             {count:>10,}   {test.loc[mask, 'ARR_DEL15'].sum():>10,}   {prop_rate:>6.1f}%")

print(f"\nBUFFER ABSORPTION ANALYSIS:")
print(f"  {'Buffer Range':<25} {'Flights':>10} {'Delayed':>10} {'Rate':>8}")
print(f"  {'-' * 56}")

delayed_incoming = test[has_prev_delay & (test['prev_tail_arr_delay'] >= 15)].copy()
buffer_bins = [(0, 30), (30, 60), (60, 90), (90, 120), (120, 300)]
for low, high in buffer_bins:
    mask = (delayed_incoming['scheduled_turnaround_buffer'] >= low) & \
           (delayed_incoming['scheduled_turnaround_buffer'] < high)
    count = mask.sum()
    if count > 0:
        rate = delayed_incoming.loc[mask, 'ARR_DEL15'].mean() * 100
        print(f"  {low:>3}-{high:<3} min buffer      {count:>10,}   {delayed_incoming.loc[mask, 'ARR_DEL15'].sum():>10,}   {rate:>6.1f}%")

print(f"\nCASCADE CHAIN LENGTH:")
print(f"  {'Cascade Score':<20} {'Flights':>10} {'Delay Rate':>12} {'Avg Delay':>12}")
print(f"  {'-' * 58}")

for score in [0, 1, 2, 3, 4, 5]:
    mask = test['cascade_score'] == score
    count = mask.sum()
    if count > 100:
        rate = test.loc[mask, 'ARR_DEL15'].mean() * 100
        avg_delay = test.loc[mask, 'ARR_DELAY'].mean()
        label = f"{score} prior delays" if score > 0 else "No prior delays"
        print(f"  {label:<20} {count:>10,}   {rate:>10.1f}%   {avg_delay:>10.1f} min")

CASCADE PROPAGATION ANALYSIS

Flights where previous aircraft was late: 1,091,450
  Of those, this flight was ALSO delayed: 525,269 (48.1%)
  Of those, this flight was ON TIME:      566,181 (51.9%)

Flights where previous aircraft was ON TIME: 1,987,157
  Of those, this flight was delayed: 253,407 (12.8%)

PROPAGATION RATE BY INCOMING DELAY SEVERITY:
  Incoming Delay               Flights   Propagated     Rate
  ----------------------------------------------------------
    0-15  min                473,569    106,095.0     22.4%
   15-30  min                220,721     95,556.0     43.3%
   30-60  min                190,896    141,400.0     74.1%
   60-120 min                130,270    116,792.0     89.7%
  120-500 min                 73,504     64,195.0     87.3%

BUFFER ABSORPTION ANALYSIS:
  Buffer Range                 Flights    Delayed     Rate
  --------------------------------------------------------
    0-30  min buffer           8,166      7,445.0     91.2%
   30-60  min buff

## Cascade Hotspots by Airport
Identify which airports experience the highest cascade activity.

In [ ]:
cascade_origins = test[test['cascade_score'] >= 1].groupby('ORIGIN').agg(
    cascade_flights=('cascade_score', 'count'),
    avg_cascade_score=('cascade_score', 'mean'),
    avg_delay=('ARR_DELAY', 'mean'),
    total_cascade_minutes=('cascade_delay_minutes', 'sum'),
).sort_values('total_cascade_minutes', ascending=False)

print(f"{'=' * 65}")
print(f"TOP 15 CASCADE SUPER-SPREADER AIRPORTS")
print(f"{'=' * 65}")
print(f"  {'Airport':<10} {'Cascade Flights':>16} {'Avg Score':>10} {'Avg Delay':>10} {'Total Cascade Min':>18}")
print(f"  {'-' * 62}")
for airport, row in cascade_origins.head(15).iterrows():
    print(f"  {airport:<10} {row['cascade_flights']:>14,}   {row['avg_cascade_score']:>8.2f}   {row['avg_delay']:>8.1f}m   {row['total_cascade_minutes']:>15,.0f}")

tail_days = test.groupby(['TAIL_NUM', 'FL_DATE']).agg(
    flights=('ARR_DEL15', 'count'),
    delays=('ARR_DEL15', 'sum'),
    total_delay_min=('ARR_DELAY', 'sum'),
    max_cascade=('cascade_score', 'max'),
    carriers=('OP_UNIQUE_CARRIER', 'first'),
).reset_index()

tail_days['delay_rate'] = tail_days['delays'] / tail_days['flights']
worst_days = tail_days[tail_days['flights'] >= 4].sort_values('total_delay_min', ascending=False)

print(f"\n{'=' * 65}")
print(f"TOP 10 WORST TAIL-NUMBER DAYS (min 4 flights)")
print(f"{'=' * 65}")
print(f"  {'Tail':<10} {'Date':<12} {'Carrier':<8} {'Flights':>8} {'Delays':>8} {'Total Min':>10} {'Max Chain':>10}")
print(f"  {'-' * 68}")
for _, row in worst_days.head(10).iterrows():
    print(f"  {row['TAIL_NUM']:<10} {str(row['FL_DATE'].date()):<12} {row['carriers']:<8} "
          f"{row['flights']:>6}   {row['delays']:>6}   {row['total_delay_min']:>8.0f}m   {row['max_cascade']:>8}")

print(f"\n{'=' * 65}")
print(f"CASCADE PROPAGATION BY CARRIER")
print(f"{'=' * 65}")
print(f"  {'Carrier':<8} {'Cascaded Flights':>16} {'Prop Rate':>10} {'Avg Cascade Min':>16}")
print(f"  {'-' * 52}")

carrier_cascade = test[test['cascade_score'] >= 1].groupby('OP_UNIQUE_CARRIER').agg(
    count=('cascade_score', 'count'),
    prop_rate=('ARR_DEL15', 'mean'),
    avg_cascade_min=('cascade_delay_minutes', 'mean'),
).sort_values('avg_cascade_min', ascending=False)

for carrier, row in carrier_cascade.iterrows():
    print(f"  {carrier:<8} {row['count']:>14,}   {row['prop_rate']:>8.1%}   {row['avg_cascade_min']:>14.1f} min")

TOP 15 CASCADE SUPER-SPREADER AIRPORTS
  Airport     Cascade Flights  Avg Score  Avg Delay  Total Cascade Min
  --------------------------------------------------------------
  DEN              43,785.0       1.15       51.3m         3,301,748
  ORD              35,656.0       1.24       46.8m         2,966,601
  DFW              35,647.0       1.22       48.7m         2,891,496
  ATL              29,039.0       1.23       52.5m         2,272,348
  CLT              21,499.0       1.29       46.1m         1,785,126
  DCA              18,373.0       1.25       56.4m         1,521,192
  LAS              20,255.0       1.29       39.2m         1,409,727
  PHX              19,047.0       1.29       33.9m         1,334,725
  LGA              13,932.0       1.14       61.6m         1,044,218
  MCO              14,356.0       1.07       66.3m         1,042,034
  LAX              15,057.0       1.20       34.9m         1,030,430
  SFO              14,614.0       1.17       35.8m           969,4

## Case Study: Worst Cascade Day
Trace a single aircraft tail through a full day of cascading delays.

In [ ]:
worst_tail = 'N524AE'
worst_date = '2025-07-26'

rotation = test[(test['TAIL_NUM'] == worst_tail) & 
                (test['FL_DATE'] == worst_date)].sort_values('DEP_HOUR')

print(f"{'=' * 70}")
print(f"CASCADE EXAMPLE: {worst_tail} on {worst_date} ({rotation['OP_UNIQUE_CARRIER'].iloc[0]})")
print(f"{'=' * 70}")
print(f"  {'#':<4} {'Route':<12} {'Dep':>5} {'Arr':>5} {'Delay':>8} {'Cascade':>8} {'Turn Gap':>10}")
print(f"  {'-' * 55}")

for i, (_, row) in enumerate(rotation.iterrows(), 1):
    route = f"{row['ORIGIN']}→{row['DEST']}"
    delay = row['ARR_DELAY']
    cascade = row['cascade_score']
    gap = row['real_time_turn_gap'] if pd.notna(row['real_time_turn_gap']) else '-'
    gap_str = f"{gap:.0f}m" if isinstance(gap, (int, float)) else gap
    print(f"  {i:<4} {route:<12} {int(row['DEP_HOUR']):>4}h {int(row['ARR_HOUR']):>4}h {delay:>+7.0f}m {cascade:>7.0f}   {gap_str:>9}")

total_delay = rotation['ARR_DELAY'].sum()
print(f"\n  Total delay accumulated: {total_delay:.0f} minutes ({total_delay/60:.1f} hours)")
print(f"  Flights delayed: {(rotation['ARR_DEL15'] == 1).sum()}/{len(rotation)}")

print(f"\n{'=' * 70}")
print(f"CASCADE ANALYSIS SUMMARY")
print(f"{'=' * 70}")
print(f"  48.1% of flights with late incoming aircraft are also delayed")
print(f"  Propagation rate: 22% (0-15 min late) → 90% (60-120 min late)")
print(f"  Cascade chains amplify: 15.7% delay rate → 86.7% after 4 cascades")
print(f"  Buffer < 30 min = 91.2% propagation vs buffer 120+ min = 40.1%")
print(f"  Top super-spreader: DEN (3.3M cascade minutes)")
print(f"  Worst carrier cascade: G4 (80.9% propagation rate)")
print(f"  Best carrier absorption: HA (47.5% propagation rate)")

CASCADE EXAMPLE: N524AE on 2025-07-26 (OH)
  #    Route          Dep   Arr    Delay  Cascade   Turn Gap
  -------------------------------------------------------
  1    DAY→ORD         8h    9h    +206m       0           -
  2    ORD→TYS         9h   12h    +184m       1       -151m
  3    TYS→ORD        13h   14h    +232m       2       -154m
  4    ORD→DAY        14h   17h   +1134m       2       -187m
  5    DAY→ORD        17h   18h   +1189m       2      -1104m
  6    ORD→TYS        18h   21h   +1535m       3      -1144m

  Total delay accumulated: 4480 minutes (74.7 hours)
  Flights delayed: 6/6

CASCADE ANALYSIS SUMMARY
  48.1% of flights with late incoming aircraft are also delayed
  Propagation rate: 22% (0-15 min late) → 90% (60-120 min late)
  Cascade chains amplify: 15.7% delay rate → 86.7% after 4 cascades
  Buffer < 30 min = 91.2% propagation vs buffer 120+ min = 40.1%
  Top super-spreader: DEN (3.3M cascade minutes)
  Worst carrier cascade: G4 (80.9% propagation rate)
  Best